In [1]:
import sympy as sp

In [2]:
class OptimizationSolver:

    def __init__(self, variables, function, decimal_places=7):
        self.variables = variables
        self.function = function
        self.decimal_places = decimal_places

        self.gradient = [
            sp.diff(function, var) for var in variables
        ]

        self.hessian = sp.hessian(function, variables)

    # 1. Tính Gradient
    def show_gradient(self):
        print("\n===== GRADIENT =====")

        for var, derivative in zip(
            self.variables, self.gradient
        ):
            print(f"df/d{var} = ", end="")
            self.display_expression(derivative)


    # 2. Tìm điểm dừng: Gradient = 0
    def find_stationary_points(self):
        print("\n===== STATIONARY POINTS =====")

        equations = [
            sp.Eq(derivative, 0)
            for derivative in self.gradient
        ]

        solutions = sp.solve(
            equations,
            self.variables,
            dict=True
        )

        if not solutions:
            print("Khong tim thay diem dung.")
            return []

        for index, solution in enumerate(solutions, 1):
            print(f"Point {index}: {solution}")

        return solutions

    # 3. Tính ma trận Hessian
    def show_hessian(self):
        print("\n===== HESSIAN MATRIX =====")
        # self.print_matrix(self.hessian)
        formatted_hessian = self.hessian.applyfunc(
            lambda value: sp.Float(
                float(value),
                self.decimal_places + 2
            )
        )

        sp.pprint(formatted_hessian)
        
    # 4. Tính các định thức con chính D1, D2, ..., Dn
    def calculate_minors(self):
        print("\n===== PRINCIPAL MINORS =====")

        minors = []

        for k in range(1, len(self.variables) + 1):
            sub_matrix = self.hessian[:k, :k]
            determinant = sp.simplify(sub_matrix.det())

            minors.append(determinant)

            print(
                f"D{k} = {self.format_number(determinant)}"
            )


        return minors

    # 5. Phân loại điểm cực trị bằng tiêu chuẩn Sylvester
    def classify_point(self, minors):
        all_positive = all(d > 0 for d in minors)

        alternating_negative = all(
            (-1) ** k * minors[k - 1] > 0
            for k in range(1, len(minors) + 1)
        )

        if all_positive:
            return "LOCAL MINIMUM"

        if alternating_negative:
            return "LOCAL MAXIMUM"

        return "INCONCLUSIVE (Check higher-order conditions)"

    # 6. Giải toàn bộ bài toán
    def solve(self):
        print("\n================================")
        print("FUNCTION:")
        # self.display_expression(self.function)
        sp.pprint(self.function)

        self.show_gradient()
        self.show_hessian()

        stationary_points = self.find_stationary_points()
        minors = self.calculate_minors()

        print("\n===== OPTIMIZATION RESULT =====")

        for point in stationary_points:
            print("\nStationary point:")

            for var in self.variables:
                print(
                    f"{var} = "
                    f"{self.format_number(point[var])}"
                )

            result = self.classify_point(minors)

            print("Classification:", result)

            function_value = self.function.subs(point)

            print(
                "f(x*) =",
                self.format_number(function_value)
            )

    def format_number(self, value):
        """Format numerical values to configurable decimal places."""
        value = sp.N(value, self.decimal_places + 5)

        if value.is_number:
            return f"{float(value):.{self.decimal_places}f}"

        return str(value)


    def print_matrix(self, matrix):
        """Display matrix with configurable decimal places."""
        formatted = matrix.applyfunc(
            lambda value: self.format_number(value)
        )
        sp.pprint(formatted)

    def format_number(self, value):
        value = sp.N(value, 20)

        if value.is_number:
            return f"{float(value):.{self.decimal_places}f}"

        return str(value)


    def format_expression(self, expression):
        """Format all numerical coefficients in an expression."""
        expression = sp.expand(expression)

        return sp.collect(
            expression,
            self.variables
        )


    def display_expression(self, expression):
        """Display expression with 7 decimal places."""
        expression = sp.N(expression, 20)

        replacements = {}

        for number in expression.atoms(sp.Number):
            replacements[number] = sp.Float(
                float(number),
                self.decimal_places + 2
            )

        expression = expression.xreplace(replacements)

        sp.pprint(expression, use_unicode=True)

In [3]:

# ==================================================
# EXAMPLE 4.10 - THREE VARIABLES
# ==================================================

x, y, z = sp.symbols("x y z")

f2 = (
    -sp.Rational(950912699, 2000000)
    + sp.Rational(2177, 20) * x
    - sp.Rational(1033, 4) * x**2
    - sp.Rational(202, 25) * y
    - 6 * x * y
    - sp.Rational(127, 10) * y**2
    + sp.Rational(37449, 100) * z
    - sp.Rational(9, 2) * x * z
    - sp.Rational(24, 5) * y * z
    - sp.Rational(1641, 20) * z**2
)

print("\n\n######## EXAMPLE 4.10 ########")

# solver2 = OptimizationSolver(
#     variables=[x, y, z],
#     function=f2
# )

# Custom: 4 decimal places

solver2 = OptimizationSolver(
    variables=[x, y, z],
    function=f2,
    decimal_places=8
)
solver2.solve()




######## EXAMPLE 4.10 ########

FUNCTION:
        2                                 2                          2         ↪
  1033⋅x            9⋅x⋅z   2177⋅x   127⋅y    24⋅y⋅z   202⋅y   1641⋅z    37449 ↪
- ─────── - 6⋅x⋅y - ───── + ────── - ────── - ────── - ───── - ─────── + ───── ↪
     4                2       20       10       5       25       20        100 ↪

↪               
↪ ⋅z   950912699
↪ ── - ─────────
↪       2000000 

===== GRADIENT =====
df/dx = -516.5⋅x - 6.0⋅y - 4.5⋅z + 108.85
df/dy = -6.0⋅x - 25.4⋅y - 4.8⋅z - 8.08
df/dz = -4.5⋅x - 4.8⋅y - 164.1⋅z + 374.49

===== HESSIAN MATRIX =====
⎡-516.5  -6.0    -4.5 ⎤
⎢                     ⎥
⎢ -6.0   -25.4   -4.8 ⎥
⎢                     ⎥
⎣ -4.5   -4.8   -164.1⎦

===== STATIONARY POINTS =====
Point 1: {x: 1/5, y: -4/5, z: 23/10}

===== PRINCIPAL MINORS =====
D1 = -516.50000000
D2 = 13083.10000000
D3 = -2134781.40000000

===== OPTIMIZATION RESULT =====

Stationary point:
x = 0.20000000
y = -0.80000000
z = 2.30000000
Classificati

In [5]:
# ==================================================
# EXAMPLE 4.1.9 - TWO VARIABLES
# ==================================================

x1, x2 = sp.symbols("x1 x2")

f1 = (
    -sp.Rational(7, 2)
    + sp.Rational(123, 10) * x1
    + x1**2 / 2
    - 41 * x2
    - 2 * x1 * x2
    + 4 * x2**2
)

print("\n\n######## EXAMPLE 4.1.9 ########")


solver1 = OptimizationSolver(
    variables=[x1, x2],
    function=f1
)

solver1.solve()




######## EXAMPLE 4.1.9 ########

FUNCTION:
  2                                       
x₁              123⋅x₁       2           7
─── - 2⋅x₁⋅x₂ + ────── + 4⋅x₂  - 41⋅x₂ - ─
 2                10                     2

===== GRADIENT =====
df/dx1 = x₁ - 2.0⋅x₂ + 12.3
df/dx2 = -2.0⋅x₁ + 8.0⋅x₂ - 41.0

===== HESSIAN MATRIX =====
⎡1.0   -2.0⎤
⎢          ⎥
⎣-2.0  8.0 ⎦

===== STATIONARY POINTS =====
Point 1: {x1: -41/10, x2: 41/10}

===== PRINCIPAL MINORS =====
D1 = 1.0000000
D2 = 4.0000000

===== OPTIMIZATION RESULT =====

Stationary point:
x1 = -4.1000000
x2 = 4.1000000
Classification: LOCAL MINIMUM
f(x*) = -112.7650000
